# Data View

In [ ]:
import pandas as pd
import sys
sys.path.append("/home/ec2-user/studies/HUIFA-RNA-SEQ-new/phenoAI/")

from phenokit.preprocessing.qc import MissingValueAnalyzer
from phenokit.preprocessing.io import read_parquet

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/liver_ultrasound/liver_ultrasound.parquet'
df_liver = read_parquet(pfile)
print(df_liver.shape)
df_liver.head()

In [ ]:
pfile = '/home/ec2-user/studies/hpp_datasets/liver_ultrasound/liver_ultrasound_aggregated.parquet'
df_liver_agg = read_parquet(pfile)
print(df_liver_agg.shape)
df_liver_agg.head()

In [ ]:
df_liver_meta = df_liver[df_liver['array_index'] == 0][['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_timestamp', 'collection_date', 'timezone']]
df_liver_agg = pd.merge(df_liver_agg, df_liver_meta, on=['participant_id', 'cohort', 'research_stage', 'array_index'])

In [ ]:
missing_analyzer = MissingValueAnalyzer(df_liver_agg)
missing_analyzer.overall_missing_rate()
missing_analyzer.report_features()
missing_analyzer.report_samples()
missing_analyzer.report_heatmap()

# Missing Value Handling

In [ ]:
liver_metadata_col = ['participant_id', 'cohort', 'research_stage', 'array_index', 'collection_timestamp', 'collection_date', 'timezone']
liver_metadata_col = [col for col in liver_metadata_col if col in df_liver_agg.columns]
df_liver_agg_metadata = df_liver_agg[liver_metadata_col]
df_liver_agg_features = df_liver_agg.drop(columns=liver_metadata_col)

In [ ]:
# 全局中位数（每列一个median）
medians = df_liver_agg_features.median()

# 用全局中位数填充缺失
df_liver_agg_features = df_liver_agg_features.fillna(medians)

# Save

In [ ]:
df_liver_agg_metadata.reset_index(drop=True, inplace=True)
df_liver_agg_features.reset_index(drop=True, inplace=True)
df_liver_agg = pd.concat([df_liver_agg_metadata, df_liver_agg_features], axis=1)
df_liver_agg.to_parquet('pheno_data/202512/004-liver_ultrasound_imputed.parquet')

In [ ]:
pfile = 'pheno_data/202512/004-liver_ultrasound_imputed.parquet'
df_liver_agg = pd.read_parquet(pfile)
print(df_liver_agg.shape)
df_liver_agg.head()